# Análisis de Cavidades en RuBisCO

Pipeline: fpocket → freesasa → APBS → Python

Basado en el paper de Poudel et al. (2020) — Biophysical Analysis of the Structural Evolution of Substrate Specificity in RuBisCO

**Ejecutar en Google Colab** (las herramientas requieren Linux/conda)

## Configuración de Colab

In [ ]:
# Clonar repo
import os
REPO_URL = "https://github.com/fpino73/analisismolecular.git"
!git clone {REPO_URL} /content/analisismolecular 2>/dev/null || (cd /content/analisismolecular && git pull)
%cd /content/analisismolecular

# Instalar dependencias Python
!pip install -q -e .
!pip install -q py3Dmol nglview plotly ipywidgets prody biopython rdkit-pypi

In [ ]:
# Instalar fpocket, freesasa, pdb2pqr, APBS via conda
!conda install -y -c conda-forge fpocket freesasa pdb2pqr apbs 2>&1 | tail -5
print("Herramientas instaladas:")
!which fpocket && fpocket -h 2>&1 | head -3
!which pdb2pqr30
!which apbs

In [ ]:
import sys
sys.path.insert(0, "/content/analisismolecular/src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from libreria_analisismolecular.cavidades import (
    run_fpocket, parse_fpocket_pockets, run_apbs,
    filter_cavities_by_charge, calculate_surface_area_per_cavity,
    plot_area_vs_s, pipeline_cavidades_rubisco
)

sns.set_style("whitegrid")
%matplotlib inline

## 1. Descargar estructuras PDB de RuBisCO

Estructuras representativas de cada grupo filogenético de los 55 datasets usado en Poudel 2020

In [ ]:
PDB_IDS = {
    "1RBO": {"group": "G-I", "organism": "Spinacia oleracea", "S": 80},
    "1IR1": {"group": "G-I", "organism": "Spinacia oleracea", "S": 80},
    "4RUB": {"group": "G-I", "organism": "Nicotiana tabacum", "S": 77},
    "1GEH": {"group": "G-II", "organism": "Rhodospirillum rubrum", "S": 15},
    "1TJY": {"group": "G-III", "organism": "Thermococcus kodakarensis", "S": 5},
}

PDB_DIR = Path("data/pdb")
PDB_DIR.mkdir(parents=True, exist_ok=True)

import urllib.request
for pdb_id in PDB_IDS:
    pdb_path = PDB_DIR / f"{pdb_id}.pdb"
    if not pdb_path.exists():
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        print(f"Descargando {pdb_id}...")
        urllib.request.urlretrieve(url, pdb_path)
    else:
        print(f"{pdb_id} ya existe")

print("\nEstructuras listas:")
for pdb_id, info in PDB_IDS.items():
    print(f"  {pdb_id} - {info['group']:5s} | {info['organism']:25s} | S={info['S']}")

## 2. Pipeline completo: fpocket → freesasa → APBS → filtro

In [ ]:
ALL_RESULTS = []

for pdb_id, info in PDB_IDS.items():
    pdb_path = PDB_DIR / f"{pdb_id}.pdb"
    if not pdb_path.exists():
        print(f"Saltando {pdb_id} (no descargado)")
        continue

    print(f"\n{'='*60}")
    print(f"Procesando {pdb_id} ({info['group']})...")
    print(f"{'='*60}")

    # 2a. fpocket: deteccion de cavidades
    print("  > fpocket: detectando cavidades...")
    fpocket_dir = run_fpocket(str(pdb_path))
    pockets = parse_fpocket_pockets(fpocket_dir)
    print(f"    Cavidades encontradas: {len(pockets)}")

    # 2b. freesasa: area superficial
    pockets = calculate_surface_area_per_cavity(pockets)

    # 2c. APBS: potencial electrostatico
    print("  > APBS: calculando potencial electrostatico...")
    apbs_dir = run_apbs(str(pdb_path))
    charged_pockets = filter_cavities_by_charge(pockets, charge_threshold=0.0)
    print(f"    Cavidades con carga positiva: {len(charged_pockets)}")

    # 2d. Anotar metadata
    charged_pockets["pdb_id"] = pdb_id
    charged_pockets["group"] = info["group"]
    charged_pockets["organism"] = info["organism"]
    charged_pockets["S_rel"] = info["S"]

    ALL_RESULTS.append(charged_pockets)

if ALL_RESULTS:
    df_all = pd.concat(ALL_RESULTS, ignore_index=True)
else:
    df_all = pd.DataFrame()

print(f"\nTotal de cavidades cargadas: {len(df_all)}")

## 3. Visualizar resultados

In [ ]:
if not df_all.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 3a. Area vs S
    sns.scatterplot(data=df_all, x="area", y="S_rel", hue="group",
                    style="group", s=100, ax=axes[0])
    axes[0].set_xlabel("Area de cavidad ($\\AA^2$)")
    axes[0].set_ylabel("Especificidad CO$_2$/O$_2$ (S)")
    axes[0].set_title("Area vs Especificidad")

    # 3b. Volumen vs S
    sns.scatterplot(data=df_all, x="volume", y="S_rel", hue="group",
                    style="group", s=100, ax=axes[1])
    axes[1].set_xlabel("Volumen de cavidad ($\\AA^3$)")
    axes[1].set_ylabel("Especificidad CO$_2$/O$_2$ (S)")
    axes[1].set_title("Volumen vs Especificidad")

    # 3c. Score fpocket vs S
    sns.scatterplot(data=df_all, x="score", y="S_rel", hue="group",
                    style="group", s=100, ax=axes[2])
    axes[2].set_xlabel("Score fpocket")
    axes[2].set_ylabel("Especificidad CO$_2$/O$_2$ (S)")
    axes[2].set_title("Score fpocket vs Especificidad")

    plt.tight_layout()
    plt.show()
else:
    print("No hay datos para graficar. Ejecuta la celda anterior primero.")

In [ ]:
# Resumen por grupo
if not df_all.empty:
    summary = df_all.groupby("group").agg({
        "area": ["mean", "std", "count"],
        "volume": ["mean", "std"],
        "score": ["mean", "std"],
        "S_rel": "first",
    }).round(2)
    display(summary)

## 4. Comparacion G-I vs G-II vs G-III

**Preguntas:**
1. G-III tiene cavidades grandes pero S baja — ¿confirma que no sigue la tendencia?
2. ¿Hay diferencia topologica (bolsas vs tunel) entre grupos?
3. ¿El score electrostatico correlaciona mejor que el area sola?

In [ ]:
if not df_all.empty:
    g = sns.catplot(data=df_all, x="group", y="area", kind="box", height=5, aspect=1.5)
    g.set_axis_labels("Grupo", "Area de cavidad ($\\AA^2$)")
    g.fig.suptitle("Distribucion de areas por grupo", y=1.02)
    plt.show()

    corr = df_all[["area", "volume", "score", "S_rel"]].corr()
    plt.figure(figsize=(6, 5))
    sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, center=0)
    plt.title("Matriz de correlacion")
    plt.show()

## 5. Conclusiones

Basado en nuestra discusion del paper Poudel 2020:

- **G-III no sigue tendencia:** cavidades grandes pero S baja — el tamano no lo explica todo
- **Cambio topologico:** G-I desarrolla un tunel continuo; G-II/G-III tienen bolsas aisladas
- **Electrostatic steering:** el gradiente cationico en el tunel orienta CO2 hacia el sitio activo
- **Implicancia:** diseno racional debe asegurar conectividad + gradiente, no solo volumen